# Topic: Logging Security Architecture - Log forging, newline sanitization, and sensitive data masking filters
 
## 1. THE LOG FORGING VULNERABILITY
- **Log Forging (Log Injection)**: Occurs when user-supplied inputs containing newline characters (`\n` or `\r`) are written to log files.
  - An attacker can inject strings like:
    `guest_user\n[INFO] 05/Jul/2026: User admin logged in successfully`
  - This injects a fake log row, allowing attackers to forge system events or deceive security auditors during post-incident investigations.
- **Mitigation**: Always sanitize input parameters before passing them to loggers by replacing carriage returns (`\r`) and line feeds (`\n`).
 
## 2. SENSITIVE DATA LEAKAGE IN LOGS
- Logging plain-text passwords, tokens, API keys, or PII (Personally Identifiable Information) violates security standards (like PCI-DSS, GDPR).
- **Mitigation (Masking Filter)**:
  - Implement a custom **`logging.Filter`** or logging formatter that parses log messages, uses regular expressions to detect sensitive patterns, and replaces them with a mask (e.g. `[MASKED]`).
 
---


In [ ]:
import logging
import re

# 1. Custom Logging Filter for Masking Sensitive Data
class SensitiveDataFilter(logging.Filter):
    """Custom logging filter that masks passwords and API tokens."""
    
    # Matches patterns like password=..., token=...
    MASK_PATTERNS = [
        (re.compile(r"(password\s*=\s*['\"]?)[^'\"\s]+(['\"]?)", re.IGNORECASE), r"\g<1>[MASKED]\g<2>"),
        (re.compile(r"(token\s*=\s*['\"]?)[^'\"\s]+(['\"]?)", re.IGNORECASE), r"\g<1>[MASKED]\g<2>")
    ]

    def filter(self, record):
        # Intercept log message and mask values in place
        msg = str(record.msg)
        for pattern, replacement in self.MASK_PATTERNS:
            msg = pattern.sub(replacement, msg)
        record.msg = msg
        return True

# Setup standard logger
logger = logging.getLogger("SecureLogger")
handler = logging.StreamHandler()
handler.addFilter(SensitiveDataFilter())
logger.addHandler(handler)
logger.setLevel(logging.INFO)

# Test sensitive log masking
print("--- Logging Sensitive Data (Masking Active) ---")
logger.info("User login attempt: username=alice, password='Password999'")
logger.info("System configuration updated: token=api-token-value-xyz")
# Expected: Passwords and tokens should show as [MASKED] in console output!



In [ ]:
# 2. Log Forging Demonstration and Sanitization
print("\n--- Log Forging / Injection Simulation ---")

def vulnerable_log_username(user_input):
    """Vulnerable logger that prints user string directly."""
    # DANGEROUS: If user_input contains newlines, it creates new log rows
    print(f"[LOG ENTRY]: User login attempt from: {user_input}")

# Normal user
vulnerable_log_username("alice_dev")

# Attack user
forging_payload = "guest_user\n[LOG ENTRY]: User admin logged in successfully from 192.168.1.50"
vulnerable_log_username(forging_payload)
# Expected: Log output shows a fake row claiming 'User admin logged in successfully'!



In [ ]:
def secure_log_username(user_input):
    """Secure logger that sanitizes newlines before logging."""
    # Sanitize: strip or replace newlines and carriage returns
    sanitized = user_input.replace("\r", "\\r").replace("\n", "\\n")
    print(f"[SECURE LOG ENTRY]: User login attempt from: {sanitized}")

print("\n--- Secure Log Output ---")
secure_log_username(forging_payload)
# Expected: The newline is escaped to '\n', keeping the entire output in a single log row!
